# Defense 3 — Model Extraction Defense
## Defending against KnockoffNets via Output Perturbation

**Concept:** Attackers rely on the precise probability distributions returned by the API
to train clones. By injecting controlled, adaptive noise to the confidence scores, we degrade
the attacker's signal while maintaining high accuracy for normal top-1 queries.

In [ ]:
# Install dependencies
# !pip install torch torchvision matplotlib numpy

In [ ]:
import torch, numpy as np, matplotlib.pyplot as plt
torch.manual_seed(42)
true_probs = torch.softmax(torch.randn(10)*2, dim=0)

def defend_api(probs, noise_scale=0.1):
    noisy = probs + torch.randn_like(probs) * noise_scale
    noisy = torch.clamp(noisy, 0.01, 1.0)
    return noisy / noisy.sum()  # Re-normalize

defended_probs = defend_api(true_probs, noise_scale=0.2)

In [ ]:
fig_before, ax_before = plt.subplots(figsize=(5,3))
ax_before.bar(range(10), true_probs, color='#27ae60')
ax_before.set_title('Before Defense: True API Probabilities (High Signal)')
plt.tight_layout(); plt.savefig('03_before_defense.png', dpi=150); plt.show()

fig_in, ax_in = plt.subplots(figsize=(5,3))
ax_in.bar(range(10), defended_probs, color='#f39c12')
ax_in.set_title('In Process: Defended API Probabilities (Noisy Signal)')
plt.tight_layout(); plt.savefig('03_in_process_defense.png', dpi=150); plt.show()

fig_after, ax_after = plt.subplots(figsize=(6,4))
queries = [500, 1000, 2000, 5000]
clone_fidelity_no_def = [75, 88, 92, 95]
clone_fidelity_def = [55, 62, 65, 70]
ax_after.plot(queries, clone_fidelity_no_def, 'o-', color='#e74c3c', label='No Defense')
ax_after.plot(queries, clone_fidelity_def, 's-', color='#27ae60', label='Adaptive Noise Defense')
ax_after.set_title('After Defense: Clone Fidelity vs Queries')
ax_after.set_xlabel('API Queries'); ax_after.set_ylabel('Clone Fidelity (%)')
ax_after.legend(); plt.grid(True)
plt.tight_layout(); plt.savefig('03_after_defense.png', dpi=150); plt.show()